In [1]:
using PowerDynamics
using PowerDynamics.Library
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as Dt
using OrdinaryDiffEqRosenbrock
using OrdinaryDiffEqNonlinearSolve
using CairoMakie
using NetworkDynamics
using DataFrames
using CSV


In [2]:
# using Pkg; Pkg.update("PowerDynamics")
pkgversion(PowerDynamics)


v"4.3.1"

In [6]:
BASE_POWER = 100e6
BASE_FREQ = 60.0
BASE_VOLTAGE = 345e3
BASE_IMPEDANCE = BASE_VOLTAGE^2 / BASE_POWER

path_to_csv_files = "dynamic-equivalents-data-export\\"

"dynamic-equivalents-data-export\\"

# Read System Data from CSV

In [7]:
using CSV

function parse_python_complex(s::String)
    s = strip(s, ['(', ')', ' '])
    m = match(r"([+-]?[\d.e+-]+)([+-][\d.e+-]+)j", s)
    if m !== nothing
        return complex(parse(Float64, m[1]), parse(Float64, m[2]))
    else
        # Handle cases like "0j"
        s = replace(s, "j" => "im")
        return parse(ComplexF64, s)
    end
end




parse_python_complex (generic function with 1 method)

## Admittance Matrix

In [8]:
admittance_matrix_pu_df = CSV.read("$(path_to_csv_files)reduced-admittance-matrix-from-power-flow.csv", DataFrame, header=1, drop=[1])
admittance_matrix_pu_df = mapcols(col -> parse_python_complex.(col), admittance_matrix_pu_df)
admittance_matrix_pu_df

Row,Cluster 1,Cluster 2,Cluster 3,Cluster 5,Cluster 7
,Complex…,Complex…,Complex…,Complex…,Complex…
1,-51.8283+25.2657im,0.0+0.0im,51.8283-25.2657im,0.0+0.0im,0.0+0.0im
2,0.0+0.0im,4.98598+13.4012im,-4.98598-13.4012im,0.0+0.0im,0.0+0.0im
3,51.8283-25.2657im,-4.98598-13.4012im,-19.5291+124.667im,-30.087-39.0915im,2.77376-46.9086im
4,0.0+0.0im,0.0+0.0im,-30.087-39.0915im,30.087+39.0915im,0.0+0.0im
5,0.0+0.0im,0.0+0.0im,2.77376-46.9086im,0.0+0.0im,-2.77376+46.9086im


In [9]:
admittance_matrix_pu = Matrix(admittance_matrix_pu_df)
admittance_matrix_pu

5×5 Matrix{ComplexF64}:
 -51.8283+25.2657im       0.0+0.0im      …       0.0+0.0im
      0.0+0.0im       4.98598+13.4012im          0.0+0.0im
  51.8283-25.2657im  -4.98598-13.4012im      2.77376-46.9086im
      0.0+0.0im           0.0+0.0im              0.0+0.0im
      0.0+0.0im           0.0+0.0im         -2.77376+46.9086im

## Power Exchange

In [10]:
power_exchange_MVA_df = CSV.read("$(path_to_csv_files)power-exchange.csv", DataFrame, header=1, drop=[1])
power_exchange_MVA_df = coalesce.(power_exchange_MVA_df, "0+0j")
power_exchange_MVA_df = mapcols(col -> parse_python_complex.(col), power_exchange_MVA_df)
power_exchange_pu_df = power_exchange_MVA_df .* 1e6 ./ BASE_POWER
power_exchange_pu_matrix = Matrix(power_exchange_pu_df)
power_exchange_pu_df

Row,Cluster 1,Cluster 2,Cluster 3,Cluster 5,Cluster 7
,Complex…,Complex…,Complex…,Complex…,Complex…
1,0.0+0.0im,0.0+0.0im,3.14195-0.552207im,0.0+0.0im,0.0+0.0im
2,0.0+0.0im,0.0+0.0im,-1.04681-0.145062im,0.0+0.0im,0.0+0.0im
3,-3.14195+0.552207im,1.04681+0.145062im,0.0+0.0im,-5.11611-1.93652im,-2.5-1.46158im
4,0.0+0.0im,0.0+0.0im,5.11611+1.93652im,0.0+0.0im,0.0+0.0im
5,0.0+0.0im,0.0+0.0im,2.5+1.46158im,0.0+0.0im,0.0+0.0im


## Load

In [11]:
load_power_MVA_df = CSV.read("$(path_to_csv_files)total-power-loads.csv", DataFrame, header=["subsystem", "total_load"], skipto=2)
load_power_MVA_df.total_load = parse_python_complex.(load_power_MVA_df.total_load)
load_power_pu_vector = Vector(load_power_MVA_df.total_load) .* 1e6 ./ BASE_POWER

5-element Vector{ComplexF64}:
                2.24 + 0.47200000762939454im
               11.04 + 2.5000000000000004im
   47.59900009155273 + 11.071000008583068im
 0.09199999809265137 + 0.045999999046325686im
                 0.0 + 0.0im

In [12]:
total_power_exchange_subsystems_pu = vec(sum(power_exchange_pu_matrix, dims=1))
total_generation_subsystems_pu = load_power_pu_vector - total_power_exchange_subsystems_pu
total_generation_subsystems_pu

5-element Vector{ComplexF64}:
  5.381952856914852 - 0.0802067040203931im
  9.993186276346366 + 2.354938410506722im
 37.887750235060295 + 8.370168682238631im
  5.208110721392797 + 1.9825178449871326im
 2.4999999999310742 + 1.4615817815466952im

In [13]:
inertia_constants = collect(x[1] for x in CSV.File("$(path_to_csv_files)total-inertia-of-synchronous-machines.csv"; select=[2])) * 1e6 / BASE_POWER

5-element Vector{Float64}:
  24.297001361846924
 500.0
 186.09499955177307
  30.30299997329712
  41.99999809265137

# Create Lines 

In [14]:
pi_line_template = MTKLine(Library.PiLine(; name=:piline))

num_buses = size(admittance_matrix_pu, 1)

# Maximum possible number of branches (upper triangle)
max_lines = (num_buses * (num_buses - 1)) ÷ 2

lines = Vector{EdgeModel}(undef, max_lines)

k = 1
@inbounds for j in 2:num_buses
    for i in 1:j-1
        Y = admittance_matrix_pu[i, j]
        if Y != 0
            Z = -inv(Y) # * BASE_IMPEDANCE # convert admittance → impedance
            lines[k] = compile_line(
                pi_line_template;
                src=i,
                dst=j,
                piline₊R=real(Z),#abs(real(Z)),
                piline₊X=imag(Z),
                name=Symbol("line_$(i)_$(j)")
            )
            k += 1
        end
    end
end

# Trim unused entries (if zeros existed)
resize!(lines, k - 1)

4-element Vector{EdgeModel}:
 EdgeModel :line_1_3 @ Edge 1=>3
 EdgeModel :line_2_3 @ Edge 2=>3
 EdgeModel :line_3_4 @ Edge 3=>4
 EdgeModel :line_3_5 @ Edge 3=>5

# Create Buses

In [15]:
@named synchronous_machine_swing = Library.Swing()
@named load_pq = PQLoad()

@named machine_load_bus_template = compile_bus(
    MTKBus(synchronous_machine_swing, load_pq);
)

strip_defaults!(machine_load_bus_template)
# set_default!(machine_load_bus_template, r"S_b$", BASE_MVA)
# set_default!(machine_load_bus_template, r"ω_b$", 2π*BASE_FREQ)



VertexModel :machine_load_bus_template NoFeedForward()
 ├─ 2 inputs:  [busbar₊i_r, busbar₊i_i]
 ├─ 2 states:  [synchronous_machine_swing₊ω≈0, synchronous_machine_swing₊θ≈0]
 ├─ 2 outputs: [busbar₊u_r, busbar₊u_i]
 └─ 7 params:  [synchronous_machine_swing₊ω_ref, synchronous_machine_swing₊Pm≈1, synchronous_machine_swing₊M, synchronous_machine_swing₊D, synchronous_machine_swing₊V≈1, load_pq₊Pset, load_pq₊Qset]

Compile buses and add power flow model. For the power flow models, the first bus is the slack and the rest are PQ controlled. Additional constraints are added to the loads for their power output. This should be enough information to solve the initialization of buses and to share as the power between the Swing and load model.

In [16]:
num_buses = size(admittance_matrix_pu, 1)

buses = Vector{VertexModel}(undef, num_buses)

for bus in 1:num_buses
    buses[bus] = compile_bus(machine_load_bus_template; vidx=bus, name=Symbol("bus_$(bus)"), synchronous_machine_swing₊ω_ref=1, synchronous_machine_swing₊M=inertia_constants[bus],
        synchronous_machine_swing₊Pm=real(total_generation_subsystems_pu[bus]), synchronous_machine_swing₊D=1e-4)
    if bus == 1
        pf_model = pfSlack(V=1, δ=0)
    else
        p_subsystem = real(total_power_exchange_subsystems_pu[bus])
        q_subsystem = imag(total_power_exchange_subsystems_pu[bus])
        pf_model = pfPQ(P=p_subsystem, Q=q_subsystem)
    end
    set_pfmodel!(buses[bus], pf_model)
    p_load = real(load_power_pu_vector[bus])
    q_load = imag(load_power_pu_vector[bus])
    constraint = @initconstraint begin
        :load_pq₊Pset - p_load
        :load_pq₊Qset - q_load
    end
    set_initconstraint!(buses[bus], constraint)
    # set_default!(buses[bus], "synchronous_machine_swing₊ω_ref", 1)
end
buses

5-element Vector{VertexModel}:
 VertexModel :bus_1 @ Vertex 1
 VertexModel :bus_2 @ Vertex 2
 VertexModel :bus_3 @ Vertex 3
 VertexModel :bus_4 @ Vertex 4
 VertexModel :bus_5 @ Vertex 5

In [17]:
buses[1]

VertexModel :bus_1 NoFeedForward() @ Vertex 1
 ├─ 2 inputs:  [busbar₊i_r, busbar₊i_i]
 ├─ 2 states:  [synchronous_machine_swing₊ω≈0, synchronous_machine_swing₊θ≈0]
 ├─ 2 outputs: [busbar₊u_r, busbar₊u_i]
 ├─ 7 params:  [synchronous_machine_swing₊ω_ref=1, synchronous_machine_swing₊Pm=5.382, synchronous_machine_swing₊M=24.297, synchronous_machine_swing₊D=0.0001, synchronous_machine_swing₊V≈1, load_pq₊Pset, load_pq₊Qset]
 └─ 2 add. init eqs. from 1 constraint for [:load_pq₊Pset, :load_pq₊Qset]
Powerflow model :slackbus with [slack₊V=1, slack₊δ=0]

# Create Network

In [18]:
nw = Network(buses, lines)

Network with 10 states and 71 parameters
 ├─ 5 vertices (1 unique type)
 └─ 4 edges (1 unique type)
Edge-Aggregation using SequentialAggregator(+)

In [19]:
pfnw = powerflow_model(nw)

Network with 8 states and 46 parameters
 ├─ 5 vertices (2 unique types)
 └─ 4 edges (1 unique type)
Edge-Aggregation using SequentialAggregator(+)

In [20]:
describe_vertices(pfnw)

Row,idx,name,batch,slack₊V,slack₊δ,pq₊P,pq₊Q,busbar₊u_i,busbar₊u_r
,Int64,Symbol,Int64,Int64?,Int64?,Float64?,Float64?,Float64?,Float64?
1,1,slackbus,1,1,0,missing,missing,missing,missing
2,2,pqbus,2,missing,missing,1.04681,0.145062,0.0,1.0
3,3,pqbus,2,missing,missing,9.71125,2.70083,0.0,1.0
4,4,pqbus,2,missing,missing,-5.11611,-1.93652,0.0,1.0
5,5,pqbus,2,missing,missing,-2.5,-1.46158,0.0,1.0


In [21]:
describe_edges(pfnw)

Row,idx,srcdst,name,batch,piline₊R,piline₊X,piline₊G_src,piline₊B_src,piline₊G_dst,piline₊B_dst,piline₊r_src,piline₊r_dst,piline₊active
,Int64,Pair…,Symbol,Int64,Float64?,Float64?,Int64?,Int64?,Int64?,Int64?,Int64?,Int64?,Int64?
1,1,1=>3,line_1_3,1,-0.0155897,-0.00759979,0,0,0,0,1,1,1
2,2,2=>3,line_2_3,1,0.0243872,-0.065547,0,0,0,0,1,1,1
3,3,3=>4,line_3_4,1,0.0123643,-0.0160647,0,0,0,0,1,1,1
4,4,3=>5,line_3_5,1,-0.00125617,-0.0212438,0,0,0,0,1,1,1


# Solve Power Flow

In [22]:
find_fixpoint(pfnw, abstol=1e-4, reltol=1e-4)

┌ Warning: The `alias_u0` keyword argument is deprecated. Please use a NonlinearAliasSpecifier, e.g. `alias = NonlinearAliasSpecifier(alias_u0 = true)`.
└ @ NonlinearSolveBase C:\Users\seberlein\.julia\packages\NonlinearSolveBase\2E600\src\solve.jl:57


NWState{Vector{Float64}} of Network (5 vertices, 4 edges)
  ├─ VIndex(2, :busbar₊u_i) => -0.09168678605325009
  ├─ VIndex(2, :busbar₊u_r) => 0.9630775643224051
  ├─ VIndex(3, :busbar₊u_i) => -0.015870483636023065
  ├─ VIndex(3, :busbar₊u_r) => 0.9536611142705225
  ├─ VIndex(4, :busbar₊u_i) => 0.09620739428996941
  ├─ VIndex(4, :busbar₊u_r) => 0.9062934762732473
  ├─ VIndex(5, :busbar₊u_i) => 0.0373511522581171
  └─ VIndex(5, :busbar₊u_r) => 0.9863103973213176
 p = NWParameter([1.0, 0.0, 1.04681, 0.145062, 9.71125, 2.70083, -5.11611, -1.93652, -2.5, -1.46158  …  1.0, -0.00125617, -0.0212438, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0])
 t = nothing

In [23]:
pfs = solve_powerflow(nw)

┌ Warning: The `alias_u0` keyword argument is deprecated. Please use a NonlinearAliasSpecifier, e.g. `alias = NonlinearAliasSpecifier(alias_u0 = true)`.
└ @ NonlinearSolveBase C:\Users\seberlein\.julia\packages\NonlinearSolveBase\2E600\src\solve.jl:57


NWState{Vector{Float64}} of Network (5 vertices, 4 edges)
  ├─ VIndex(2, :busbar₊u_i) => -0.0916853191261014
  ├─ VIndex(2, :busbar₊u_r) => 0.9630793268863227
  ├─ VIndex(3, :busbar₊u_i) => -0.01586959273104796
  ├─ VIndex(3, :busbar₊u_r) => 0.9536623708882426
  ├─ VIndex(4, :busbar₊u_i) => 0.09620825945019808
  ├─ VIndex(4, :busbar₊u_r) => 0.9062932349284146
  ├─ VIndex(5, :busbar₊u_i) => 0.037351861501018546
  └─ VIndex(5, :busbar₊u_r) => 0.9863112957938287
 p = NWParameter([1.0, 0.0, 1.04681, 0.145062, 9.71125, 2.70083, -5.11611, -1.93652, -2.5, -1.46158  …  1.0, -0.00125617, -0.0212438, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0])
 t = nothing

# Initialize Dynami Models

In [24]:
dyn_bus_model = nw[VIndex(1)]
dump_initial_state(dyn_bus_model; obs=false)

Inputs:
  busbar₊i_i                      =  uninitialized           
  busbar₊i_r                      =  uninitialized           
States:
  synchronous_machine_swing₊θ     =  uninitialized (guess  0)
  synchronous_machine_swing₊ω     =  uninitialized (guess  0)
Outputs:
  busbar₊u_i                      =  uninitialized           
  busbar₊u_r                      =  uninitialized           
Parameters:
  load_pq₊Pset                    =  uninitialized           
  load_pq₊Qset                    =  uninitialized           
  synchronous_machine_swing₊D     =  0.0001                  
  synchronous_machine_swing₊M     =  24.297                  
  synchronous_machine_swing₊Pm    =  5.382         (guess  1)
  synchronous_machine_swing₊V     =  uninitialized (guess  1)
  synchronous_machine_swing₊ω_ref =  1                       


In [25]:
powerflow_model(dyn_bus_model)

VertexModel :slackbus NoFeedForward()
 ├─ 2 inputs:  [busbar₊i_r, busbar₊i_i]
 ├─ 0 states:  []
 ├─ 2 outputs: [busbar₊u_r=1, busbar₊u_i=0]
 └─ 2 params:  [slack₊V=1, slack₊δ=0]

In [27]:
s0 = initialize_from_pf!(nw)

┌ Warning: The `alias_u0` keyword argument is deprecated. Please use a NonlinearAliasSpecifier, e.g. `alias = NonlinearAliasSpecifier(alias_u0 = true)`.
└ @ NonlinearSolveBase C:\Users\seberlein\.julia\packages\NonlinearSolveBase\2E600\src\solve.jl:57


Initializing vertex 1...

LoadError: ComponentInitError: Missing guesses for free variables [:load_pq₊Pset, :load_pq₊Qset]